# 03 · EDA temporal y preparación del panel de demanda

## Objetivo

El clustering identificó el **Cluster 4** como el segmento estratégico inicial para la PoC: mercados con alto volumen, múltiples proveedores y diferencias económicas relevantes.

El objetivo de este notebook es transformar esos mercados en un panel temporal fiable para forecasting:

> **Semana + Tratamiento + Ciudad → Número de servicios**

Antes de entrenar modelos se resuelven cuatro cuestiones metodológicas:

1. Incorporar correctamente la fecha del servicio.
2. Trabajar únicamente con semanas completas.
3. Representar explícitamente semanas sin demanda mediante `0`, sin inventar ceros antes de que un mercado aparezca.
4. Distinguir entre **ausencia real de demanda** y una posible **ruptura estructural / descontinuación**.

El resultado será un panel semanal definitivo que servirá de entrada al notebook de modelado.

## 0. Archivos requeridos

Este notebook utiliza tres insumos:

- `datos tratramiento detalleV4.xlsx`: histórico principal.
- `datos tratramiento detalleV4_fecha.xlsx`: fecha asociada a `IDEPREADMIN + NUMLIQUID`.
- `mercados_cluster_k5.csv`: segmentación generada por el notebook 02.
- `calendario_venezuela_ml_2025_2027.xlsx`: calendario semanal para variables exógenas.

> En Google Colab, carga estos archivos en `/content/` o ajusta las rutas de la siguiente celda.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

def buscar_archivo(nombre):
    candidatos = [
        Path("/content") / nombre,
        Path("/mnt/data") / nombre,
        Path(nombre)
    ]
    return next((p for p in candidatos if p.exists()), None)

DATA_PATH = buscar_archivo("datos tratramiento detalleV4.xlsx")
DATE_PATH = buscar_archivo("datos tratramiento detalleV4_fecha.xlsx")
CLUSTER_PATH = buscar_archivo("mercados_cluster_k5.csv")
CALENDAR_PATH = buscar_archivo("calendario_venezuela_ml_2025_2027.xlsx")

archivos = {
    "Histórico": DATA_PATH,
    "Fechas": DATE_PATH,
    "Clusters": CLUSTER_PATH,
    "Calendario": CALENDAR_PATH
}

for nombre, ruta in archivos.items():
    print(f"{nombre:12}: {ruta}")

faltantes = [nombre for nombre, ruta in archivos.items() if ruta is None]

if faltantes:
    raise FileNotFoundError(
        "Faltan archivos requeridos: "
        + ", ".join(faltantes)
        + ". Cárgalos en Colab o ajusta las rutas."
    )

## 1. Carga del histórico y selección del Cluster 4

In [ ]:
df = pd.read_excel(DATA_PATH)
df = df[df["SUMA"] > 0].copy()

mercados_competencia = pd.read_csv(CLUSTER_PATH)

mercados_cluster4 = (
    mercados_competencia.loc[
        mercados_competencia["CLUSTER_5"] == 4,
        ["CODIGO_TRATAMIENTO", "CODIGO_MUNICIPIO"]
    ]
    .drop_duplicates()
)

print(f"Servicios históricos válidos: {len(df):,}")
print(f"Mercados Cluster 4: {len(mercados_cluster4):,}")

display(
    mercados_competencia[
        mercados_competencia["CLUSTER_5"] == 4
    ][
        [
            "CODIGO_TRATAMIENTO",
            "DESCPROCED",
            "CODIGO_MUNICIPIO",
            "DESCMUNICIPIO",
            "SERVICIOS",
            "PROVEEDORES",
            "COSTO_MEDIANA",
            "BRECHA_PCT"
        ]
    ]
    .sort_values("SERVICIOS", ascending=False)
    .head(15)
)

### Segmento seleccionado

El Cluster 4 contiene **51 mercados Tratamiento + Ciudad**. Se selecciona como PoC porque combina:

- alto volumen;
- múltiples proveedores;
- alta proporción de volumen atendido por proveedores con evidencia;
- diferencias económicas relevantes.

La siguiente etapa ya no analiza proveedores individualmente: ahora se pregunta **cuánta demanda semanal tendrá cada mercado**.

## 2. Incorporación y validación de la fecha

In [ ]:
df_fecha = pd.read_excel(DATE_PATH)

print(f"Filas tabla de fechas: {len(df_fecha):,}")
print(f"Columnas: {df_fecha.columns.tolist()}")

df_fecha["FECOCURRLIQ"] = pd.to_datetime(
    df_fecha["FECOCURRLIQ"],
    dayfirst=True,
    errors="coerce"
)

print("Fechas nulas:", df_fecha["FECOCURRLIQ"].isna().sum())
print("Fecha mínima:", df_fecha["FECOCURRLIQ"].min())
print("Fecha máxima:", df_fecha["FECOCURRLIQ"].max())

In [ ]:
# La tabla de fechas tiene granularidad IDEPREADMIN + NUMLIQUID.
diagnostico_fecha = (
    df_fecha
    .groupby(
        ["IDEPREADMIN", "NUMLIQUID"],
        dropna=False
    )
    .agg(
        REGISTROS=("FECOCURRLIQ", "size"),
        FECHAS_DISTINTAS=("FECOCURRLIQ", "nunique")
    )
    .reset_index()
)

print(
    "Llaves repetidas:",
    (diagnostico_fecha["REGISTROS"] > 1).sum()
)

print(
    "Llaves con fechas diferentes:",
    (diagnostico_fecha["FECHAS_DISTINTAS"] > 1).sum()
)

if (diagnostico_fecha["FECHAS_DISTINTAS"] > 1).any():
    display(
        diagnostico_fecha[
            diagnostico_fecha["FECHAS_DISTINTAS"] > 1
        ].head(20)
    )

In [ ]:
# Eliminamos duplicados únicamente después de validar que una misma
# llave no tenga fechas contradictorias.
df_fecha = (
    df_fecha
    .drop_duplicates(
        subset=["IDEPREADMIN", "NUMLIQUID"]
    )
    .copy()
)

df_final = df.merge(
    df_fecha[
        ["IDEPREADMIN", "NUMLIQUID", "FECOCURRLIQ"]
    ],
    on=["IDEPREADMIN", "NUMLIQUID"],
    how="inner"
)

cobertura_fecha = len(df_final) / len(df) * 100

print(f"Servicios antes del join: {len(df):,}")
print(f"Servicios con fecha: {len(df_final):,}")
print(f"Cobertura de fecha: {cobertura_fecha:.2f}%")

### Hallazgo

En el desarrollo original se obtuvo una cobertura temporal cercana al **99,5 %**. La unión se realiza mediante `INNER JOIN` porque la tabla de fechas contiene registros adicionales de producción que no pertenecen necesariamente al histórico analizado.

No se imputan fechas faltantes: un servicio sin fecha no puede participar de forma fiable en una serie temporal.

## 3. Construcción de la semana y selección temporal del Cluster 4

In [ ]:
df_final["FECHA_SEMANA"] = (
    df_final["FECOCURRLIQ"]
    .dt.to_period("W-SUN")
    .apply(lambda x: x.start_time)
)

df_cluster4 = df_final.merge(
    mercados_cluster4,
    on=["CODIGO_TRATAMIENTO", "CODIGO_MUNICIPIO"],
    how="inner"
)

print(f"Servicios Cluster 4 con fecha: {len(df_cluster4):,}")
print(
    "Mercados preservados:",
    df_cluster4[
        ["CODIGO_TRATAMIENTO", "CODIGO_MUNICIPIO"]
    ].drop_duplicates().shape[0]
)
print("Fecha mínima:", df_cluster4["FECOCURRLIQ"].min())
print("Fecha máxima:", df_cluster4["FECOCURRLIQ"].max())

## 4. Eliminación de semanas de borde incompletas

La frecuencia de trabajo es semanal, con semanas **lunes–domingo**.

Si la fuente comienza después de un lunes o termina antes de un domingo, la primera/última semana representa solo una fracción del periodo. Esas semanas no deben utilizarse para entrenamiento ni evaluación porque aparentarían una caída artificial de demanda.

In [ ]:
fecha_min_raw = df_cluster4["FECOCURRLIQ"].min()
fecha_max_raw = df_cluster4["FECOCURRLIQ"].max()

semana_min = df_cluster4["FECHA_SEMANA"].min()
semana_max = df_cluster4["FECHA_SEMANA"].max()

primera_incompleta = fecha_min_raw.normalize() > semana_min
ultima_incompleta = fecha_max_raw.normalize() < (semana_max + pd.Timedelta(days=6))

print("Primera semana:", semana_min.date(), "| incompleta:", primera_incompleta)
print("Última semana :", semana_max.date(), "| incompleta:", ultima_incompleta)

fecha_inicio_completa = (
    semana_min + pd.Timedelta(weeks=1)
    if primera_incompleta
    else semana_min
)

fecha_fin_completa = (
    semana_max - pd.Timedelta(weeks=1)
    if ultima_incompleta
    else semana_max
)

print("Ventana semanal completa:")
print(fecha_inicio_completa.date(), "→", fecha_fin_completa.date())

## 5. Demanda semanal observada

In [ ]:
columnas_serie = [
    "CODIGO_TRATAMIENTO",
    "DESCPROCED",
    "CODIGO_MUNICIPIO",
    "DESCMUNICIPIO"
]

# La llave de servicio se mantiene para evitar duplicación accidental.
llave_servicio = [
    "IDEPREADMIN",
    "NUMLIQUID",
    "CODIGO_TRATAMIENTO"
]

servicios_cluster4 = (
    df_cluster4[
        (df_cluster4["FECHA_SEMANA"] >= fecha_inicio_completa) &
        (df_cluster4["FECHA_SEMANA"] <= fecha_fin_completa)
    ]
    .drop_duplicates(subset=llave_servicio)
    .copy()
)

demanda_observada = (
    servicios_cluster4
    .groupby(
        ["FECHA_SEMANA"] + columnas_serie
    )
    .size()
    .reset_index(name="N_SERVICIOS")
)

demanda_observada["SERIE_ID"] = (
    demanda_observada["CODIGO_TRATAMIENTO"].astype(str)
    + "_"
    + demanda_observada["CODIGO_MUNICIPIO"].astype(str)
)

print(f"Registros semana-mercado observados: {len(demanda_observada):,}")
print(f"Servicios agregados: {demanda_observada['N_SERVICIOS'].sum():,}")
print(f"Series: {demanda_observada['SERIE_ID'].nunique():,}")

## 6. Panel inicial: ceros únicamente entre primera y última aparición

No se rellenan ceros desde el inicio global para todas las series porque eso podría inventar ausencia de demanda **antes de que un tratamiento existiera en una ciudad**.

Como diagnóstico inicial, cada mercado se completa entre:

`PRIMERA_SEMANA_POSITIVA → ULTIMA_SEMANA_POSITIVA`

Las semanas sin servicios dentro de ese intervalo se registran como `0`.

In [ ]:
paneles_iniciales = []

for serie_id, datos_serie in demanda_observada.groupby("SERIE_ID"):

    datos_serie = datos_serie.sort_values("FECHA_SEMANA").copy()

    semanas = pd.DataFrame({
        "FECHA_SEMANA": pd.date_range(
            datos_serie["FECHA_SEMANA"].min(),
            datos_serie["FECHA_SEMANA"].max(),
            freq="W-MON"
        )
    })

    meta = datos_serie.iloc[0]

    for col in columnas_serie:
        semanas[col] = meta[col]

    semanas["SERIE_ID"] = serie_id

    semanas = semanas.merge(
        datos_serie[["FECHA_SEMANA", "N_SERVICIOS"]],
        on="FECHA_SEMANA",
        how="left"
    )

    semanas["N_SERVICIOS"] = (
        semanas["N_SERVICIOS"].fillna(0).astype(int)
    )

    paneles_iniciales.append(semanas)

df_panel_inicial = (
    pd.concat(paneles_iniciales, ignore_index=True)
    .sort_values(["SERIE_ID", "FECHA_SEMANA"])
    .reset_index(drop=True)
)

print(f"Filas panel inicial: {len(df_panel_inicial):,}")
print(f"Series: {df_panel_inicial['SERIE_ID'].nunique():,}")
print(
    "Semanas con demanda cero:",
    f"{df_panel_inicial['N_SERVICIOS'].eq(0).mean():.2%}"
)

## 7. EDA temporal agregado

In [ ]:
diagnostico_semanal = (
    df_panel_inicial
    .groupby("FECHA_SEMANA")
    .agg(
        SERVICIOS=("N_SERVICIOS", "sum"),
        MERCADOS_ACTIVOS=(
            "N_SERVICIOS",
            lambda x: (x > 0).sum()
        ),
        MERCADOS_TOTAL=("N_SERVICIOS", "size")
    )
    .reset_index()
)

diagnostico_semanal["PCT_MERCADOS_ACTIVOS"] = (
    diagnostico_semanal["MERCADOS_ACTIVOS"]
    / diagnostico_semanal["MERCADOS_TOTAL"]
    * 100
)

diagnostico_semanal["SERVICIOS_POR_MERCADO"] = (
    diagnostico_semanal["SERVICIOS"]
    / diagnostico_semanal["MERCADOS_TOTAL"]
)

diagnostico_semanal["MEDIA_MOVIL_4"] = (
    diagnostico_semanal["SERVICIOS"]
    .rolling(4, min_periods=1)
    .mean()
)

display(
    diagnostico_semanal
    .sort_values("SERVICIOS")
    .head(15)
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    diagnostico_semanal["FECHA_SEMANA"],
    diagnostico_semanal["SERVICIOS"],
    marker="o",
    markersize=3,
    label="Servicios semanales"
)

ax.plot(
    diagnostico_semanal["FECHA_SEMANA"],
    diagnostico_semanal["MEDIA_MOVIL_4"],
    linewidth=2,
    label="Media móvil 4 semanas"
)

ax.set_title("Demanda semanal total — Cluster 4")
ax.set_xlabel("Semana")
ax.set_ylabel("Servicios")
ax.grid(alpha=0.2)
ax.legend()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

ax.plot(
    diagnostico_semanal["FECHA_SEMANA"],
    diagnostico_semanal["PCT_MERCADOS_ACTIVOS"],
    marker="o",
    markersize=3
)

ax.set_title("Porcentaje de mercados activos por semana — Cluster 4")
ax.set_xlabel("Semana")
ax.set_ylabel("% mercados con demanda")
ax.grid(alpha=0.2)

plt.show()

### Hallazgos temporales

Durante el desarrollo se identificaron caídas transversales en periodos de calendario conocidos, especialmente:

- Navidad y fin de año.
- Inicio de enero.
- Semana Santa.

El hecho de que la caída afecte simultáneamente a numerosos mercados sugiere un **efecto calendario común**, no un comportamiento aislado de un tratamiento.

Por esta razón el modelo incorporará posteriormente variables exógenas de calendario conocidas antes del momento de predicción.

## 8. Comportamiento de las series de mayor volumen

In [ ]:
resumen_series_inicial = (
    df_panel_inicial
    .groupby(
        ["SERIE_ID"] + columnas_serie
    )
    .agg(
        SEMANAS=("FECHA_SEMANA", "size"),
        SEMANAS_CON_DEMANDA=(
            "N_SERVICIOS",
            lambda x: (x > 0).sum()
        ),
        SERVICIOS=("N_SERVICIOS", "sum"),
        DEMANDA_MEDIA=("N_SERVICIOS", "mean"),
        DEMANDA_MEDIANA=("N_SERVICIOS", "median"),
        DEMANDA_MAX=("N_SERVICIOS", "max")
    )
    .reset_index()
)

resumen_series_inicial["PCT_SEMANAS_CERO"] = (
    1
    - resumen_series_inicial["SEMANAS_CON_DEMANDA"]
    / resumen_series_inicial["SEMANAS"]
)

display(
    resumen_series_inicial[
        [
            "SEMANAS",
            "SEMANAS_CON_DEMANDA",
            "SERVICIOS",
            "DEMANDA_MEDIA",
            "DEMANDA_MEDIANA",
            "DEMANDA_MAX",
            "PCT_SEMANAS_CERO"
        ]
    ].describe().T
)

In [ ]:
top4 = (
    resumen_series_inicial
    .nlargest(4, "SERVICIOS")["SERIE_ID"]
    .tolist()
)

for serie_id in top4:

    serie = (
        df_panel_inicial[
            df_panel_inicial["SERIE_ID"] == serie_id
        ]
        .sort_values("FECHA_SEMANA")
        .copy()
    )

    serie["MEDIA_MOVIL_4"] = (
        serie["N_SERVICIOS"]
        .rolling(4, min_periods=1)
        .mean()
    )

    fig, ax = plt.subplots(figsize=(12, 3.8))

    ax.plot(
        serie["FECHA_SEMANA"],
        serie["N_SERVICIOS"],
        marker="o",
        markersize=3,
        label="Demanda semanal"
    )

    ax.plot(
        serie["FECHA_SEMANA"],
        serie["MEDIA_MOVIL_4"],
        linewidth=2,
        label="Media móvil 4 semanas"
    )

    ax.set_title(
        f"{serie['DESCPROCED'].iloc[0]} — "
        f"{serie['DESCMUNICIPIO'].iloc[0]}"
    )
    ax.set_xlabel("Semana")
    ax.set_ylabel("Servicios")
    ax.grid(alpha=0.2)
    ax.legend()

    plt.show()

## 9. Diagnóstico de patrones de demanda: ADI y CV²

Se utilizan dos métricas descriptivas:

- **ADI (Average Demand Interval):** cuántas semanas transcurren, en promedio, por cada semana con demanda positiva.
- **CV²:** variabilidad relativa del tamaño de la demanda cuando ésta ocurre.

Los umbrales descriptivos tradicionales son:

- `ADI = 1,32`
- `CV² = 0,49`

Clasificación:

| Tipo | ADI | CV² | Interpretación |
|---|---|---|---|
| Smooth | bajo | bajo | frecuente y relativamente estable |
| Erratic | bajo | alto | frecuente pero variable |
| Intermittent | alto | bajo | muchos ceros, tamaño relativamente estable |
| Lumpy | alto | alto | muchos ceros y tamaños variables |

> Esta clasificación se utiliza para **EDA y evaluación posterior**, no como feature del modelo.

In [ ]:
def calcular_adi_cv2(grupo):

    demanda = (
        grupo
        .sort_values("FECHA_SEMANA")["N_SERVICIOS"]
        .to_numpy()
    )

    demanda_positiva = demanda[demanda > 0]

    adi = (
        len(demanda) / len(demanda_positiva)
        if len(demanda_positiva) > 0
        else np.nan
    )

    if len(demanda_positiva) > 1 and demanda_positiva.mean() > 0:
        cv2 = (
            demanda_positiva.std(ddof=1)
            / demanda_positiva.mean()
        ) ** 2
    else:
        cv2 = 0.0

    return pd.Series({
        "ADI": adi,
        "CV2": cv2
    })

metricas_demanda_inicial = (
    df_panel_inicial
    .groupby(["SERIE_ID"] + columnas_serie)
    .apply(calcular_adi_cv2, include_groups=False)
    .reset_index()
)

def clasificar_demanda(row):
    if row["ADI"] < 1.32 and row["CV2"] < 0.49:
        return "SMOOTH"
    if row["ADI"] < 1.32 and row["CV2"] >= 0.49:
        return "ERRATIC"
    if row["ADI"] >= 1.32 and row["CV2"] < 0.49:
        return "INTERMITTENT"
    return "LUMPY"

metricas_demanda_inicial["TIPO_DEMANDA"] = (
    metricas_demanda_inicial
    .apply(clasificar_demanda, axis=1)
)

metricas_demanda_inicial = metricas_demanda_inicial.merge(
    resumen_series_inicial[
        [
            "SERIE_ID",
            "SERVICIOS",
            "PCT_SEMANAS_CERO",
            "DEMANDA_MEDIA"
        ]
    ],
    on="SERIE_ID",
    how="left"
)

resumen_tipo_inicial = (
    metricas_demanda_inicial
    .groupby("TIPO_DEMANDA")
    .agg(
        MERCADOS=("SERIE_ID", "size"),
        SERVICIOS=("SERVICIOS", "sum"),
        ADI_MEDIANA=("ADI", "median"),
        CV2_MEDIANA=("CV2", "median"),
        PCT_CEROS_MEDIANA=("PCT_SEMANAS_CERO", "median"),
        DEMANDA_MEDIA=("DEMANDA_MEDIA", "mean")
    )
    .reset_index()
)

resumen_tipo_inicial["PCT_MERCADOS"] = (
    resumen_tipo_inicial["MERCADOS"]
    / resumen_tipo_inicial["MERCADOS"].sum()
    * 100
)

resumen_tipo_inicial["PCT_SERVICIOS"] = (
    resumen_tipo_inicial["SERVICIOS"]
    / resumen_tipo_inicial["SERVICIOS"].sum()
    * 100
)

display(
    resumen_tipo_inicial
    .sort_values("SERVICIOS", ascending=False)
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for tipo in metricas_demanda_inicial["TIPO_DEMANDA"].unique():

    tmp = metricas_demanda_inicial[
        metricas_demanda_inicial["TIPO_DEMANDA"] == tipo
    ]

    ax.scatter(
        tmp["ADI"],
        tmp["CV2"],
        label=tipo,
        s=60,
        alpha=0.7
    )

ax.axvline(1.32, linestyle="--")
ax.axhline(0.49, linestyle="--")

ax.set_xlabel("ADI — Intervalo medio entre demandas")
ax.set_ylabel("CV² — Variabilidad del tamaño de demanda")
ax.set_title("Patrones de demanda — Cluster 4")
ax.grid(alpha=0.2)
ax.legend()

plt.show()

### Lectura de negocio

El Cluster 4 no es temporalmente homogéneo. Conviven mercados de demanda frecuente con mercados intermitentes.

Esto justifica:

- mantener métricas robustas ante ceros;
- utilizar lags y medias móviles;
- incorporar calendario;
- evaluar el modelo también por tipo de demanda.

Sin embargo, **no se fuerza un algoritmo distinto por tipo de demanda**: la selección final se realizará mediante backtesting.

## 10. Problema detectado: truncamiento al final de las series

In [ ]:
ultima_semana_global = df_panel_inicial["FECHA_SEMANA"].max()

cobertura_final = (
    df_panel_inicial
    .groupby(["SERIE_ID"] + columnas_serie)
    .agg(
        PRIMERA_SEMANA=("FECHA_SEMANA", "min"),
        ULTIMA_SEMANA=("FECHA_SEMANA", "max"),
        SERVICIOS=("N_SERVICIOS", "sum")
    )
    .reset_index()
)

cobertura_final["SEMANAS_HASTA_FINAL"] = (
    (
        ultima_semana_global
        - cobertura_final["ULTIMA_SEMANA"]
    ).dt.days / 7
).astype(int)

print(
    "Series que llegan hasta la última semana:",
    (cobertura_final["SEMANAS_HASTA_FINAL"] == 0).sum()
)
print(
    "Series que terminan antes:",
    (cobertura_final["SEMANAS_HASTA_FINAL"] > 0).sum()
)
print(
    "Series que terminan >=4 semanas antes:",
    (cobertura_final["SEMANAS_HASTA_FINAL"] >= 4).sum()
)
print(
    "Series que terminan >=8 semanas antes:",
    (cobertura_final["SEMANAS_HASTA_FINAL"] >= 8).sum()
)

display(
    cobertura_final
    .sort_values("SEMANAS_HASTA_FINAL", ascending=False)
    .head(15)
)

### Por qué importa

Si una serie sigue vigente pero no tuvo servicios en las últimas semanas, esas semanas deben existir como `0`.

De lo contrario:

`última demanda → la serie desaparece`

cuando la realidad sería:

`última demanda → 0 → 0 → 0 ...`

Este truncamiento altera lags, medias móviles, tasas de cero y la cobertura del backtesting.

## 11. Diagnóstico de gaps terminales

In [ ]:
series_revisar = set(
    cobertura_final.loc[
        cobertura_final["SEMANAS_HASTA_FINAL"] >= 4,
        "SERIE_ID"
    ]
)

resultado_gaps = []

for serie_id in series_revisar:

    serie = (
        df_panel_inicial[
            df_panel_inicial["SERIE_ID"] == serie_id
        ]
        .sort_values("FECHA_SEMANA")
        .copy()
    )

    positivas = serie[serie["N_SERVICIOS"] > 0].copy()

    gaps = (
        positivas["FECHA_SEMANA"]
        .diff()
        .dt.days
        .div(7)
        .dropna()
    )

    info = cobertura_final[
        cobertura_final["SERIE_ID"] == serie_id
    ].iloc[0]

    resultado_gaps.append({
        "SERIE_ID": serie_id,
        "TRATAMIENTO": info["DESCPROCED"],
        "CIUDAD": info["DESCMUNICIPIO"],
        "SERVICIOS": info["SERVICIOS"],
        "GAP_FINAL": info["SEMANAS_HASTA_FINAL"],
        "GAP_HIST_MEDIANA": gaps.median() if len(gaps) else np.nan,
        "GAP_HIST_P90": gaps.quantile(.90) if len(gaps) else np.nan,
        "GAP_HIST_MAX": gaps.max() if len(gaps) else np.nan
    })

diagnostico_gaps = (
    pd.DataFrame(resultado_gaps)
    .sort_values("GAP_FINAL", ascending=False)
)

# Heurística conservadora para detectar posibles rupturas:
# gap terminal mayor al máximo histórico y, además, superior a 8 semanas.
diagnostico_gaps["CANDIDATA_RUPTURA"] = (
    diagnostico_gaps["GAP_FINAL"]
    > diagnostico_gaps[["GAP_HIST_MAX"]].fillna(0).max(axis=1)
) & (
    diagnostico_gaps["GAP_FINAL"] > 8
)

display(diagnostico_gaps)

### Hallazgo de ruptura estructural

Durante el desarrollo apareció un caso claramente diferente al resto:

**ECOSONOGRAMA ABDOMINAL — LIBERTADOR**

- 103 servicios históricos.
- Gap final: **48 semanas**.
- Gap histórico mediano: **1 semana**.
- Gap histórico máximo: **2 semanas**.

Una ausencia de 48 semanas no es compatible con su comportamiento histórico. Se interpreta como posible:

- descontinuación del código;
- cambio de contratación;
- cambio de codificación;
- o salida del mercado.

Por tanto, **no se inventan 48 semanas en cero** para esta serie.

En producción, la vigencia deberá provenir del catálogo/baremo actual y no inferirse únicamente del histórico.

## 12. Construcción del panel definitivo V2

In [ ]:
# Las candidatas a ruptura no se extienden automáticamente.
series_no_extender = set(
    diagnostico_gaps.loc[
        diagnostico_gaps["CANDIDATA_RUPTURA"],
        "SERIE_ID"
    ]
)

print("Series no extendidas:")
print(series_no_extender)

paneles_v2 = []

for serie_id, datos_serie in demanda_observada.groupby("SERIE_ID"):

    datos_serie = (
        datos_serie
        .sort_values("FECHA_SEMANA")
        .copy()
    )

    primera_semana = datos_serie["FECHA_SEMANA"].min()

    if serie_id in series_no_extender:
        ultima_semana = datos_serie["FECHA_SEMANA"].max()
    else:
        ultima_semana = fecha_fin_completa

    fechas = pd.DataFrame({
        "FECHA_SEMANA": pd.date_range(
            start=primera_semana,
            end=ultima_semana,
            freq="W-MON"
        )
    })

    meta = datos_serie.iloc[0]

    for col in columnas_serie:
        fechas[col] = meta[col]

    fechas["SERIE_ID"] = serie_id

    fechas = fechas.merge(
        datos_serie[["FECHA_SEMANA", "N_SERVICIOS"]],
        on="FECHA_SEMANA",
        how="left"
    )

    fechas["N_SERVICIOS"] = (
        fechas["N_SERVICIOS"]
        .fillna(0)
        .astype(int)
    )

    paneles_v2.append(fechas)

df_semanal_base_v2 = (
    pd.concat(paneles_v2, ignore_index=True)
    .sort_values(["SERIE_ID", "FECHA_SEMANA"])
    .reset_index(drop=True)
)

print(f"Filas panel inicial : {len(df_panel_inicial):,}")
print(f"Filas panel V2      : {len(df_semanal_base_v2):,}")
print(f"Series históricas   : {df_semanal_base_v2['SERIE_ID'].nunique():,}")
print(
    "Semanas con cero  :",
    f"{df_semanal_base_v2['N_SERVICIOS'].eq(0).mean():.2%}"
)

## 13. Validaciones críticas del panel V2

In [ ]:
assert (
    demanda_observada["N_SERVICIOS"].sum()
    == df_semanal_base_v2["N_SERVICIOS"].sum()
), "El panel alteró el número total de servicios."

assert (
    demanda_observada["SERIE_ID"].nunique()
    == df_semanal_base_v2["SERIE_ID"].nunique()
), "Se perdieron series en la construcción del panel."

series_ultima_semana = set(
    df_semanal_base_v2.loc[
        df_semanal_base_v2["FECHA_SEMANA"] == fecha_fin_completa,
        "SERIE_ID"
    ]
)

series_totales = set(
    df_semanal_base_v2["SERIE_ID"].unique()
)

print(
    "Servicios observados:",
    f"{demanda_observada['N_SERVICIOS'].sum():,}"
)
print(
    "Servicios panel V2:",
    f"{df_semanal_base_v2['N_SERVICIOS'].sum():,}"
)
print(
    "Series totales:",
    len(series_totales)
)
print(
    "Series presentes en última semana:",
    len(series_ultima_semana)
)
print(
    "Series que no llegan al final:",
    series_totales - series_ultima_semana
)

### Resultado esperado del desarrollo

En la versión final construida durante la PoC:

- El panel pasó de **3.694 a 3.745 filas**.
- Se añadieron **51 semanas-serie** con demanda cero.
- El porcentaje de semanas cero quedó alrededor de **31,08 %**.
- Se mantuvieron **51 series históricas**.
- **50 series** quedaron vigentes para forecast al final del periodo.
- La única serie no extendida fue `ECOSONOGRAMA ABDOMINAL — LIBERTADOR`.

La corrección es pequeña en tamaño, pero metodológicamente importante porque evita que la ausencia de demanda se confunda con desaparición de la serie.

## 14. Recalcular el diagnóstico ADI/CV² sobre el panel definitivo

In [ ]:
resumen_series_v2 = (
    df_semanal_base_v2
    .groupby(["SERIE_ID"] + columnas_serie)
    .agg(
        SEMANAS=("FECHA_SEMANA", "size"),
        SEMANAS_CON_DEMANDA=(
            "N_SERVICIOS",
            lambda x: (x > 0).sum()
        ),
        SERVICIOS=("N_SERVICIOS", "sum"),
        DEMANDA_MEDIA=("N_SERVICIOS", "mean"),
        DEMANDA_MEDIANA=("N_SERVICIOS", "median"),
        DEMANDA_MAX=("N_SERVICIOS", "max")
    )
    .reset_index()
)

resumen_series_v2["PCT_SEMANAS_CERO"] = (
    1
    - resumen_series_v2["SEMANAS_CON_DEMANDA"]
    / resumen_series_v2["SEMANAS"]
)

metricas_demanda_v2 = (
    df_semanal_base_v2
    .groupby(["SERIE_ID"] + columnas_serie)
    .apply(calcular_adi_cv2, include_groups=False)
    .reset_index()
)

metricas_demanda_v2["TIPO_DEMANDA"] = (
    metricas_demanda_v2.apply(clasificar_demanda, axis=1)
)

metricas_demanda_v2 = metricas_demanda_v2.merge(
    resumen_series_v2[
        [
            "SERIE_ID",
            "SEMANAS",
            "SEMANAS_CON_DEMANDA",
            "SERVICIOS",
            "DEMANDA_MEDIA",
            "DEMANDA_MEDIANA",
            "DEMANDA_MAX",
            "PCT_SEMANAS_CERO"
        ]
    ],
    on="SERIE_ID",
    how="left"
)

resumen_tipo_v2 = (
    metricas_demanda_v2
    .groupby("TIPO_DEMANDA")
    .agg(
        MERCADOS=("SERIE_ID", "size"),
        SERVICIOS=("SERVICIOS", "sum"),
        ADI_MEDIANA=("ADI", "median"),
        CV2_MEDIANA=("CV2", "median"),
        PCT_CEROS_MEDIANA=("PCT_SEMANAS_CERO", "median"),
        DEMANDA_MEDIA=("DEMANDA_MEDIA", "mean")
    )
    .reset_index()
)

resumen_tipo_v2["PCT_MERCADOS"] = (
    resumen_tipo_v2["MERCADOS"]
    / resumen_tipo_v2["MERCADOS"].sum()
    * 100
)

resumen_tipo_v2["PCT_SERVICIOS"] = (
    resumen_tipo_v2["SERVICIOS"]
    / resumen_tipo_v2["SERVICIOS"].sum()
    * 100
)

display(resumen_tipo_v2.sort_values("SERVICIOS", ascending=False))

## 15. Incorporación del calendario semanal

Las caídas transversales identificadas en el EDA justifican incorporar información conocida **antes** de realizar el forecast.

Variables disponibles:

- número de festivos;
- festivos de lunes a viernes;
- días laborables;
- periodo vacacional de referencia;
- Carnaval;
- Semana Santa;
- Navidad / fin de año.

No se crea una variable retrospectiva tipo `SEMANA_BAJA`, porque se derivaría del propio target y produciría leakage.

In [ ]:
calendario = pd.read_excel(
    CALENDAR_PATH,
    sheet_name="Calendario_Semanal"
)

calendario["FECHA_SEMANA"] = pd.to_datetime(
    calendario["FECHA_SEMANA"]
)

variables_calendario = [
    "N_FESTIVOS_TOTAL",
    "N_FESTIVOS_LV",
    "DIAS_LABORABLES_LV",
    "N_DIAS_VACACIONALES_REF",
    "PCT_DIAS_VACACIONALES_REF",
    "TIENE_FESTIVO",
    "TIENE_VACACIONAL_REF",
    "ES_SEMANA_SANTA",
    "ES_CARNAVAL",
    "ES_NAVIDAD_FIN_ANIO",
    "SEMANA_COMPLETA"
]

df_panel_final = df_semanal_base_v2.merge(
    calendario[
        ["FECHA_SEMANA"] + variables_calendario
    ],
    on="FECHA_SEMANA",
    how="left"
)

display(
    df_panel_final[variables_calendario]
    .isna()
    .sum()
    .to_frame("NULOS")
)

## 16. Validación visual de efectos de calendario

El siguiente resumen no pretende demostrar causalidad. Sirve para comprobar si los periodos de calendario identificados también presentan diferencias descriptivas en la demanda agregada.

In [ ]:
demanda_calendario = (
    df_panel_final
    .groupby("FECHA_SEMANA")
    .agg(
        SERVICIOS=("N_SERVICIOS", "sum"),
        MERCADOS=("SERIE_ID", "size"),
        ES_SEMANA_SANTA=("ES_SEMANA_SANTA", "max"),
        ES_CARNAVAL=("ES_CARNAVAL", "max"),
        ES_NAVIDAD_FIN_ANIO=("ES_NAVIDAD_FIN_ANIO", "max")
    )
    .reset_index()
)

demanda_calendario["SERVICIOS_POR_MERCADO"] = (
    demanda_calendario["SERVICIOS"]
    / demanda_calendario["MERCADOS"]
)

periodos = {
    "Semana Santa": "ES_SEMANA_SANTA",
    "Carnaval": "ES_CARNAVAL",
    "Navidad / fin de año": "ES_NAVIDAD_FIN_ANIO"
}

resumen_calendario = []

for nombre, columna in periodos.items():
    con_evento = demanda_calendario.loc[
        demanda_calendario[columna] == 1,
        "SERVICIOS_POR_MERCADO"
    ]
    sin_evento = demanda_calendario.loc[
        demanda_calendario[columna] == 0,
        "SERVICIOS_POR_MERCADO"
    ]

    resumen_calendario.append({
        "PERIODO": nombre,
        "SEMANAS_EVENTO": len(con_evento),
        "MEDIA_SERVICIOS_POR_MERCADO_EVENTO": con_evento.mean(),
        "MEDIA_RESTO": sin_evento.mean()
    })

display(pd.DataFrame(resumen_calendario))

## 17. Dataset de salida para modelado

In [ ]:
OUTPUT_PANEL = Path("/mnt/data/panel_cluster4_v2.csv")
OUTPUT_METRICAS = Path("/mnt/data/metricas_demanda_cluster4_v2.csv")

df_panel_final.to_csv(
    OUTPUT_PANEL,
    index=False
)

metricas_demanda_v2.to_csv(
    OUTPUT_METRICAS,
    index=False
)

print(f"Panel temporal exportado: {OUTPUT_PANEL}")
print(f"Métricas por serie exportadas: {OUTPUT_METRICAS}")

print("\nDimensión panel final:")
print(df_panel_final.shape)
print(
    "Periodo:",
    df_panel_final["FECHA_SEMANA"].min(),
    "→",
    df_panel_final["FECHA_SEMANA"].max()
)
print(
    "Series:",
    df_panel_final["SERIE_ID"].nunique()
)

# Conclusiones

El análisis temporal deja preparado el insumo de forecasting con una semántica consistente:

1. La demanda se modelará semanalmente a nivel **Tratamiento + Ciudad**.
2. Se eliminan semanas de borde incompletas para no introducir caídas artificiales.
3. Los ceros dentro de una serie representan ausencia real de servicios.
4. No se inventan ceros antes de la primera aparición del mercado.
5. Los gaps terminales se revisan para distinguir demanda cero de posibles rupturas estructurales.
6. El panel definitivo mantiene **51 series históricas**, pero solo las vigentes deberán participar del forecast actual.
7. Existen patrones Smooth e Intermittent, por lo que las métricas de evaluación deben tolerar ceros.
8. Se incorporan variables de calendario porque se observaron efectos transversales alrededor de periodos festivos.
9. ADI/CV² se utiliza como diagnóstico y **no como feature**, evitando incorporar estadísticas calculadas sobre todo el histórico dentro del entrenamiento.

El siguiente notebook construirá las features temporales, definirá targets H1–H4, aplicará rolling backtesting y comparará los modelos candidatos.

➡️ **Siguiente notebook: `04_Modelado_Validacion.ipynb`**